<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_05_05_03_01_deep_causal_ml_continuous_optimization_models_notears_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1visd1lH1fbGOM1qqlYJgPaubUt2srmme)


![](https://raw.githubusercontent.com/zia207/Causal_Inference_R/main/Markdown/Image/05-03-deep-CausalML-structural-learning.png)

# 5.3.1 Continuous Optimization Models (NOTEARS)

**NOTEARS** ("DAGs with NO TEARS", Zheng et al., NeurIPS 2018) introduced a landmark reformulation of causal structure learning: instead of exhaustively searching the combinatorial space of $2^{O(d^2)}$ possible directed acyclic graphs (an approach that becomes intractable for more than roughly ten variables), it recasts the problem as a **smooth, gradient-based optimization** over a real-valued weighted adjacency matrix $\mathbf{W} \in \mathbb{R}^{d \times d}$.

Under the **linear Structural Equation Model (SEM)** assumption

$$\mathbf{X} = \mathbf{X}\mathbf{W} + \mathbf{Z},$$

where $\mathbf{Z}$ contains independent noise terms (commonly Gaussian), the algorithm simultaneously (i) fits $\mathbf{W}$ to the data via a reconstruction loss, (ii) enforces acyclicity through a closed-form differentiable constraint, and (iii) promotes sparsity with an $\ell_1$ penalty.

**Key innovation — the acyclicity constraint.** The function

$$h(\mathbf{W}) = \mathrm{tr}\!\left(\exp(\mathbf{W} \odot \mathbf{W})\right) - d$$

equals zero *if and only if* the graph encoded by $\mathbf{W}$ is acyclic. Because $h$ is smooth and differentiable, the NP-hard combinatorial constraint is replaced by an equality constraint amenable to augmented Lagrangian (AL) optimization with L-BFGS inner steps — no combinatorial search required.

NOTEARS is concise (the core algorithm fits in fewer than 60 lines of code), fast for moderate graph sizes ($d \leq 100$–$200$), and has become the canonical baseline against which all later continuous causal discovery methods are benchmarked — including GOLEM, DAG-GNN, and GraN-DAG.

![](https://raw.githubusercontent.com/zia207/Causal_Inference_R/main/Markdown/Image/NOTEARS.png)


## NOTEARS with {RCausalML}

This notebook implements both the **linear** and **nonlinear** NOTEARS variants in R using the {RCausalML} package. We apply them to synthetic data (linear and nonlinear SEMs) and evaluate performance against known ground-truth DAGs.

The **RCausalML** package encodes the full NOTEARS family. The acyclicity constraint $h(W) = \mathrm{tr}(e^{W \odot W}) - d = 0$ is enforced via the trace of the matrix exponential, and optimization is handled by the augmented Lagrangian method with L-BFGS-B for linear models and Adam followed by L-BFGS for nonlinear models.

**Linear NOTEARS** learns a weighted adjacency matrix $W$ by minimizing a loss (least-squares, logistic, or Poisson) plus an $\ell_1$ penalty subject to $h(W) = 0$; the matrix exponential is computed with the `expm` package.

**Nonlinear NOTEARS** (requires **torch**) replaces the linear predictor with either an MLP or a Sobolev-basis model, allowing the algorithm to capture arbitrary smooth nonlinear relationships while keeping the same acyclicity constraint.

### Main Functions

| Function | Role |
|------------------------------------|------------------------------------|
| `notears_linear` | Linear NOTEARS: L2/logistic/Poisson loss + L1, augmented Lagrangian |
| `notears_nonlinear` | Nonlinear NOTEARS with MLP or Sobolev model (Adam then L-BFGS) |
| `NotearsMLP` | Torch module: MLP for nonlinear SEM |
| `NotearsSobolev` | Torch module: Sobolev basis for nonlinear SEM |

### Simulation and Evaluation Utilities

| Function | Role |
|------------------------------------|------------------------------------|
| `is_dag` | Check whether a weighted adjacency matrix encodes a DAG |
| `simulate_dag` | Simulate a random DAG (Erdős–Rényi or scale-free) |
| `simulate_parameter` | Draw edge weights for a DAG |
| `simulate_linear_sem` | Generate data from a linear SEM (Gaussian, exponential, uniform noise) |
| `simulate_nonlinear_sem` | Generate data from a nonlinear SEM (MLP or MIM) |
| `count_accuracy` | Compare an estimated graph to ground truth (FDR, TPR, FPR, SHD, nnz) |
| `set_random_seed` | Set R and torch seeds for reproducibility |

**Reference:** Zheng, X., Aragam, B., Ravikumar, P., & Xing, E. P. (2018). DAGs with NO TEARS: Continuous optimization for structure learning. *NeurIPS*. <https://arxiv.org/abs/1803.01422>.


## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y

!pip install rpy2==3.5.1

%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Set Up

### Check and Install Required R Packages

Following R packages are required to run this notebook. If any of these packages are not installed, you can install them using the code below:

`tidyverse`, `plyr`, `causaldata`, `torch`, `expm`, `ggdag`, `igraph`, `reshape2`, `gridExtra`, `RCausalML`


In [ ]:
%%R
packages <- c(
  "tidyverse",
  "plyr",
  "causaldata",
  "torch",
  "expm",
  "ggdag",
  "igraph",
  "reshape2",
  "gridExtra",
  "RCausalML"
)


### Install Missing Packages


In [ ]:
%%R
# Install missing packages
# new_packages <- packages[!(packages %in% installed.packages()[, "Package"])]
# if (length(new_packages)) install.packages(new_packages)


### Verify Installation


In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load R Packages


In [ ]:
%%R
# Load packages with suppressed messages
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))


### Check Loaded Packages


In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


## Part I: Linear NOTEARS on Synthetic Gaussian Data

### Overview

We begin with a **linear Gaussian SEM**: variables are generated by the model $\mathbf{X} = \mathbf{X}\mathbf{W} + \mathbf{Z}$, where $\mathbf{Z} \sim \mathcal{N}(0, I)$. The ground-truth DAG is known, so we can measure recovery accuracy precisely. This setting is the native domain of linear NOTEARS — the best-case scenario against which nonlinear variants will later be compared.


### Data Generation and Preprocessing


In [ ]:
%%R
# ── Dataset 1: Linear Gaussian SEM (d = 20) ──────────────────────────────────
set_random_seed(42)

n_lin <- 2000; d_lin <- 20; s0_lin <- 20
graph_type <- "ER"; sem_type_lin <- "gauss"

# Simulate true DAG, edge weights, and observed data
B_true_lin <- simulate_dag(d_lin, s0_lin, graph_type)
W_true_lin <- simulate_parameter(B_true_lin)
X_lin      <- simulate_linear_sem(W_true_lin, n_lin, sem_type_lin)

# Persist for reproducibility
write.csv(X_lin,      "X_linear.csv",      row.names = FALSE)
write.csv(W_true_lin, "W_true_linear.csv", row.names = FALSE)

# Center columns — NOTEARS is sensitive to variable scale
X_lin <- scale(X_lin, center = TRUE, scale = FALSE)

# 70 / 15 / 15 train / validation / test split
set.seed(42)
tr1  <- sample(1:n_lin, floor(0.70 * n_lin))
tmp1 <- setdiff(1:n_lin, tr1)
va1  <- sample(tmp1, floor(0.50 * length(tmp1)))
te1  <- setdiff(tmp1, va1)

X_lin_train <- X_lin[tr1, ]
X_lin_val   <- X_lin[va1, ]
X_lin_test  <- X_lin[te1, ]

cat("Linear SEM — Train:", dim(X_lin_train),
    "| Val:", dim(X_lin_val), "| Test:", dim(X_lin_test), "\n")


### Fitting Linear NOTEARS

The algorithm minimizes the augmented Lagrangian

$$\mathcal{L}_\rho(W, \alpha) = \frac{1}{2n}\|X - XW\|_F^2 + \lambda_1\|W\|_1 + \alpha\, h(W) + \frac{\rho}{2}h(W)^2,$$

iterating (i) an L-BFGS-B step on $W$ and (ii) a dual-ascent update $\alpha \leftarrow \alpha + \rho\, h(W)$ until $h(W) < \epsilon$.


In [ ]:
%%R
W_est_lin <- notears_linear(
  X           = X_lin_train,
  lambda1     = 0.1,
  loss_type   = "l2",
  max_iter    = 100,
  h_tol       = 1e-8,
  rho_max     = 1e+16,
  w_threshold = 0.3
)

cat("Acyclic (linear SEM):", is_dag(W_est_lin), "\n")

# Validation reconstruction loss: (1/(2n)) ||X_val - X_val W||_F^2
n_val_lin    <- nrow(X_lin_val)
val_loss_lin <- 0.5 / n_val_lin * sum((X_lin_val - X_lin_val %*% W_est_lin)^2)
cat(sprintf("Validation L2 loss (linear model | linear SEM): %.6f\n", val_loss_lin))

B_est_lin <- (abs(W_est_lin) > 0) * 1


**Hyperparameter guidance.** The two most impactful choices are: (i) the sparsity penalty $\lambda_1 \in [0.01, 0.5]$ — higher values prune more edges, trading TPR for FDR; and (ii) the edge-threshold $w_{\mathrm{threshold}} \in [0.1, 0.5]$ applied after optimization. Cross-validate on SHD when a held-out ground-truth graph is available. For very large $d$, prefer `notears_nonlinear()` with the MLP model.


### Causal Effect Estimation via Backdoor Adjustment

NOTEARS recovers **graph structure**, not causal effects directly. Once the DAG is in hand, downstream causal inference proceeds in two steps: (i) read off a valid adjustment set from the graph using the backdoor criterion, and (ii) estimate the Average Treatment Effect (ATE) or Conditional Average Treatment Effect (CATE) by regression on that set.

The example below treats variable 1 as the treatment $T$ and variable 20 as the outcome $Y$.


In [ ]:
%%R
# ── True ATE via structural path sum: (I − W)^{-1}[Y, T] ──────────────────
I_lin          <- diag(d_lin)
true_total_lin <- solve(I_lin - W_true_lin)
true_ATE_lin   <- true_total_lin[20, 1]   # R is 1-indexed
cat(sprintf("True ATE (T → Y, linear SEM):  %.4f\n", true_ATE_lin))

# ── Backdoor-adjusted OLS estimate ─────────────────────────────────────────
# All variables except T and Y serve as the adjustment set
confounders_idx_lin <- setdiff(1:d_lin, c(1, 20))
T_tr_lin  <- X_lin_train[, 1]
Y_tr_lin  <- X_lin_train[, 20]
C_tr_lin  <- X_lin_train[, confounders_idx_lin]

model_lin   <- lm(Y_tr_lin ~ T_tr_lin + C_tr_lin)
est_ATE_lin <- coef(model_lin)["T_tr_lin"]
cat(sprintf("Estimated ATE:  %.4f  (error: %.4f)\n",
            est_ATE_lin, abs(est_ATE_lin - true_ATE_lin)))

# ── Heterogeneous CATE via interaction model ───────────────────────────────
X_int_lin      <- model.matrix(~ T_tr_lin * C_tr_lin - 1)
model_cate_lin <- lm(Y_tr_lin ~ X_int_lin - 1)
cat("CATE interaction model fitted. Use predict(model_cate_lin, newdata) for individual predictions.\n")


### Graph Recovery Metrics

When the ground-truth DAG is known, five complementary metrics quantify structural recovery accuracy.

-   **FDR (False Discovery Rate)** — fraction of predicted edges that are spurious: $\text{FDR} = \text{FP} / (\text{TP} + \text{FP})$. Lower is better.
-   **TPR (True Positive Rate / Recall)** — fraction of true edges that are recovered: $\text{TPR} = \text{TP} / (\text{TP} + \text{FN})$. Higher is better.
-   **FPR (False Positive Rate)** — fraction of true non-edges that are incorrectly predicted as edges: $\text{FPR} = \text{FP} / (\text{FP} + \text{TN})$. Lower is better.
-   **SHD (Structural Hamming Distance)** — total number of edge insertions, deletions, and reversals needed to convert the estimated graph into the true graph. Lower is better; SHD = 0 means perfect recovery. This is the primary summary statistic reported in causal discovery benchmarks.
-   **nnz (Non-Zero entries)** — the number of edges in the estimated graph. Compare with the true edge count: a large discrepancy indicates over- or under-sparsification.


In [ ]:
%%R
acc_lin_result <- count_accuracy(B_true_lin, B_est_lin)
print(acc_lin_result)   # list(fdr = , tpr = , fpr = , shd = , nnz = )


An SHD below 5–10 is considered excellent for a 20-variable graph; TPR above 0.8 with FDR below 0.2 indicates strong recovery.


### Visualization

#### Weighted Adjacency Matrix

The estimated matrix $W \in \mathbb{R}^{d \times d}$ encodes both the **topology** (which entries are non-zero) and **strength** (the magnitude and sign) of each directed edge. Entry $W[i, j]$ is the estimated effect of cause $j$ on effect $i$: positive values indicate excitatory relationships, negative values inhibitory ones. A column of zeros means that variable has no detected children; a row of zeros means it has no detected parents (a root node).


In [ ]:
%%R
W_melt_lin <- melt(W_est_lin)
colnames(W_melt_lin) <- c("Effect", "Cause", "Weight")

ggplot(W_melt_lin, aes(x = Cause, y = Effect, fill = Weight)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red",
                       midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Learned Weighted Adjacency Matrix — Linear SEM (NOTEARS Linear)",
       x = "Cause (source)", y = "Effect (destination)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1))


#### Learned Causal DAG

The binary thresholded version of $W$ is rendered as a directed graph. Nodes represent variables; edges indicate estimated direct causal relationships. Compare the structure visually with the heatmap above.


In [ ]:
%%R
G_lin <- graph_from_adjacency_matrix(W_est_lin != 0, mode = "directed", weighted = TRUE)
V(G_lin)$name <- 1:d_lin
pos_lin <- spring_layout(G_lin, seed = 42)

plot(G_lin,
     layout             = pos_lin,
     vertex.color       = "lightblue",
     vertex.label.color = "black",
     edge.color         = "gray40",
     edge.arrow.size    = 0.5,
     main               = "Learned Causal DAG — Linear Gaussian SEM (NOTEARS Linear)",
     vertex.size        = 15,
     vertex.label.cex   = 0.8)


## Part II: Nonlinear NOTEARS (MLP and Sobolev) vs. Misspecified Baseline

### Overview

We now stress-test the linear model on data generated from a **nonlinear SEM** and compare it against two nonlinear NOTEARS variants. The goal is to quantify the cost of the linearity assumption and demonstrate when the more expressive models are warranted.

The three variants evaluated are:

1.  **Linear NOTEARS** (misspecified baseline) — the same algorithm as Part I, applied naively to nonlinear data.
2.  **NotearsMLP** — replaces the linear SEM with a per-variable MLP $f_i(\mathbf{X}_{\mathrm{pa}(i)})$ parameterized by a torch neural network; acyclicity is enforced on the induced binary adjacency weights.
3.  **NotearsSobolev** — replaces the MLP with a Sobolev-basis expansion (truncated sine series), providing a smooth nonparametric alternative that is especially competitive when the true functions are periodic or band-limited.


### Custom Nonlinear SEM Simulator


In [ ]:
%%R
# ── Nonlinear SEM simulator ───────────────────────────────────────────────────
# Convention: W[i, j] = weight of directed edge j → i
#             (row = destination / effect, column = source / cause)
#
# sem_type:
#   "mlp"  – tanh(z)       smooth, bounded nonlinearity
#   "mim"  – z * sin(z)    multiplicative interaction (non-monotone)
#   "gp"   – sin(z)        periodic / Gaussian-process-like
#
simulate_nonlinear_sem_custom <- function(W, n, sem_type = "mlp", noise_sd = 0.5) {
  d <- ncol(W)

  # igraph adjacency convention: A[a, b] != 0 => edge a -> b
  # Since W[i, j] != 0 <=> j -> i, we need A = t(W)
  g   <- igraph::graph_from_adjacency_matrix((t(W) != 0) * 1L, mode = "directed")
  ord <- as.integer(igraph::topo_sort(g))   # parents before children

  X <- matrix(0.0, nrow = n, ncol = d)

  for (i in ord) {
    pa <- which(W[i, ] != 0)               # parents of node i

    if (length(pa) == 0L) {
      X[, i] <- rnorm(n)                   # root node: Gaussian noise
    } else {
      z <- as.vector(X[, pa, drop = FALSE] %*% W[i, pa])  # weighted parent sum
      X[, i] <- switch(sem_type,
        mlp = tanh(z)    + rnorm(n, sd = noise_sd),
        mim = z * sin(z) + rnorm(n, sd = noise_sd),
        gp  = sin(z)     + rnorm(n, sd = noise_sd),
                 z       + rnorm(n)          # fallback: linear
      )
    }
  }
  X
}


### Data Generation and Preprocessing


In [ ]:
%%R
# ── Dataset 2: Nonlinear SEM (d = 10) ─────────────────────────────────────────
# d = 10 keeps NotearsMLP / Sobolev runtimes tractable in a notebook
set_random_seed(42)
n_nlin <- 2000; d_nlin <- 10; s0_nlin <- 10
B_true_nlin <- simulate_dag(d_nlin, s0_nlin, graph_type)
W_true_nlin <- simulate_parameter(B_true_nlin)
X_nlin      <- simulate_nonlinear_sem_custom(W_true_nlin, n_nlin, sem_type = "mlp")

write.csv(X_nlin,      "X_nonlinear.csv",      row.names = FALSE)
write.csv(W_true_nlin, "W_true_nonlinear.csv", row.names = FALSE)

X_nlin <- scale(X_nlin, center = TRUE, scale = FALSE)

set.seed(42)
tr2  <- sample(1:n_nlin, floor(0.70 * n_nlin))
tmp2 <- setdiff(1:n_nlin, tr2)
va2  <- sample(tmp2, floor(0.50 * length(tmp2)))
te2  <- setdiff(tmp2, va2)

X_nlin_train <- X_nlin[tr2, ]
X_nlin_val   <- X_nlin[va2, ]
X_nlin_test  <- X_nlin[te2, ]

cat("Nonlinear SEM — Train:", dim(X_nlin_train),
    "| Val:", dim(X_nlin_val), "| Test:", dim(X_nlin_test), "\n")


### Model 1: Linear NOTEARS (Misspecified Baseline)


In [ ]:
%%R
W_lin_nlin <- notears_linear(
  X           = X_nlin_train,
  lambda1     = 0.1,
  loss_type   = "l2",
  max_iter    = 100,
  h_tol       = 1e-8,
  rho_max     = 1e+16,
  w_threshold = 0.3
)

cat("Linear NOTEARS on nonlinear SEM — acyclic:", is_dag(W_lin_nlin), "\n")

n_val_nlin      <- nrow(X_nlin_val)
val_loss_lin_nl <- 0.5 / n_val_nlin *
                   sum((X_nlin_val - X_nlin_val %*% W_lin_nlin)^2)
cat(sprintf("Validation L2 loss (linear model | nonlinear SEM): %.6f\n", val_loss_lin_nl))

B_lin_nlin <- (abs(W_lin_nlin) > 0) * 1


### Model 2: NotearsMLP

The MLP variant parameterizes each conditional $f_i(\mathbf{X}_{\mathrm{pa}(i)})$ with a two-layer neural network with tanh activations. The Adam optimizer handles the initial stochastic descent; `lbfgs_iter > 0` switches to L-BFGS for the final convergence phase. A training loss curve is plotted when the model stores iteration history.


In [ ]:
%%R
torch_set_default_dtype(torch_double())
set_random_seed(42)

model_mlp_nlin <- NotearsMLP(d = d_nlin, hidden = 10, bias = TRUE)

W_mlp_nlin <- notears_nonlinear(
  model_mlp_nlin, X_nlin_train,
  lambda1     = 0.01,
  lambda2     = 0.01,
  max_iter    = 150L,
  lbfgs_iter  = 0L,
  w_threshold = 0.3,
  lr          = 1e-2,
  verbose     = FALSE
)

cat("NotearsMLP on nonlinear SEM — acyclic:", is_dag(W_mlp_nlin), "\n")
B_mlp_nlin <- (abs(W_mlp_nlin) > 1e-5) * 1

# Training loss curve
losses_mlp <- attr(W_mlp_nlin, "train_losses")
if (!is.null(losses_mlp) && length(losses_mlp) > 0) {
  ggplot(data.frame(Iteration = seq_along(losses_mlp), Loss = losses_mlp),
         aes(x = Iteration, y = Loss)) +
    geom_line(color = "steelblue", linewidth = 0.8) +
    labs(title = "NotearsMLP — Training Loss (Nonlinear SEM)",
         x = "Iteration", y = "Loss") +
    theme_minimal()
}


### Model 3: NotearsSobolev

The Sobolev variant expands each causal function in a truncated sine basis of order $k$. This acts as a nonparametric smoother: when $k$ is small the model stays close to linear, and as $k$ grows it can represent increasingly complex smooth functions. It is typically faster to train than the MLP and works well when the underlying relationships are smooth or periodic.


In [ ]:
%%R
set_random_seed(42)

# k = 4 basis functions per variable (sine-based Sobolev expansion)
model_sob_nlin <- NotearsSobolev(d = d_nlin, k = 4L)

W_sob_nlin <- notears_nonlinear(
  model_sob_nlin, X_nlin_train,
  lambda1     = 0.01,
  lambda2     = 0.01,
  max_iter    = 150L,
  lbfgs_iter  = 0L,
  w_threshold = 0.3,
  lr          = 1e-2,
  verbose     = FALSE
)

cat("NotearsSobolev on nonlinear SEM — acyclic:", is_dag(W_sob_nlin), "\n")
B_sob_nlin <- (abs(W_sob_nlin) > 1e-5) * 1

# Training loss curve
losses_sob <- attr(W_sob_nlin, "train_losses")
if (!is.null(losses_sob) && length(losses_sob) > 0) {
  ggplot(data.frame(Iteration = seq_along(losses_sob), Loss = losses_sob),
         aes(x = Iteration, y = Loss)) +
    geom_line(color = "darkorange", linewidth = 0.8) +
    labs(title = "NotearsSobolev — Training Loss (Nonlinear SEM)",
         x = "Iteration", y = "Loss") +
    theme_minimal()
}


### Model Comparison


In [ ]:
%%R
# Accuracy helper — skips metric computation if the graph is not a valid DAG
safe_acc <- function(B_true, W_est, B_est, label) {
  if (!is_dag(W_est)) {
    cat(sprintf("  %-30s  NOT a DAG — skipping metrics\n", label))
    return(NULL)
  }
  count_accuracy(B_true, B_est)
}

acc_lin_on_nlin <- safe_acc(B_true_nlin, W_lin_nlin, B_lin_nlin, "Linear NOTEARS (nonlin SEM)")
acc_mlp_on_nlin <- safe_acc(B_true_nlin, W_mlp_nlin, B_mlp_nlin, "NotearsMLP")
acc_sob_on_nlin <- safe_acc(B_true_nlin, W_sob_nlin, B_sob_nlin, "NotearsSobolev")

print_row <- function(label, acc) {
  if (is.null(acc)) {
    cat(sprintf("  %-34s %7s %7s %7s %5s %5s\n", label, "--", "--", "--", "--", "--"))
  } else {
    cat(sprintf("  %-34s %7.4f %7.4f %7.4f %5d %5d\n",
                label, acc$fdr, acc$tpr, acc$fpr, acc$shd, acc$nnz))
  }
}

div <- paste(rep("-", 72), collapse = "")
cat("\n", div, "\n")
cat(sprintf("  %-34s %7s %7s %7s %5s %5s\n", "Model", "FDR", "TPR", "FPR", "SHD", "nnz"))
cat(div, "\n")

# Reference: linear model on its native linear SEM
cat(sprintf("  %-34s %7.4f %7.4f %7.4f %5d %5d   (d=20, linear SEM)\n",
            "Linear NOTEARS — native SEM *",
            acc_lin_result$fdr, acc_lin_result$tpr, acc_lin_result$fpr,
            acc_lin_result$shd, acc_lin_result$nnz))
cat(div, "\n")

# All three models on nonlinear SEM
print_row("Linear NOTEARS  (misspecified baseline)", acc_lin_on_nlin)
print_row("NotearsMLP      (hidden = 10)",           acc_mlp_on_nlin)
print_row("NotearsSobolev  (k = 4)",                 acc_sob_on_nlin)
cat(div, "\n")
cat(sprintf("  True edges in nonlinear SEM graph: %d  |  d = %d\n",
            sum(B_true_nlin), d_nlin))
cat("  * different d and SEM — shown for qualitative reference only\n")
cat("  Lower FDR / FPR / SHD and higher TPR = better\n\n")


### ATE Estimation on the Nonlinear SEM

We select the best-performing valid DAG from the comparison above (MLP if available, then Sobolev, then linear as fallback), derive the adjustment set as the parents of $Y$ in the estimated graph, and estimate the ATE of variable 1 (treatment $T$) on variable 10 (outcome $Y$) via OLS. A 500-replicate bootstrap provides a 95% confidence interval.

Note that $(I - W)^{-1}$ gives exact total effects only for linear SEMs; in the nonlinear case the structural ATE shown below is a linear approximation and should be treated as an order-of-magnitude reference rather than a precise estimate.


In [ ]:
%%R
T_idx_nlin <- 1L
Y_idx_nlin <- d_nlin   # last variable as outcome

# Select best available DAG for the adjustment set
if (!is.null(acc_mlp_on_nlin)) {
  W_best_nlin  <- W_mlp_nlin;  label_best <- "NotearsMLP"
} else if (!is.null(acc_sob_on_nlin)) {
  W_best_nlin  <- W_sob_nlin;  label_best <- "NotearsSobolev"
} else {
  W_best_nlin  <- W_lin_nlin;  label_best <- "Linear NOTEARS (fallback)"
}
cat(sprintf("Adjustment set derived from: %s\n", label_best))

# Adjustment set: parents of Y in the learned graph, excluding T
pa_Y_nlin <- which(abs(W_best_nlin[Y_idx_nlin, ]) > 1e-5)
conf_nlin  <- setdiff(pa_Y_nlin, T_idx_nlin)
if (length(conf_nlin) == 0L)
  conf_nlin <- setdiff(1:d_nlin, c(T_idx_nlin, Y_idx_nlin))  # fallback: all others

T_n <- X_nlin_train[, T_idx_nlin]
Y_n <- X_nlin_train[, Y_idx_nlin]
C_n <- X_nlin_train[, conf_nlin, drop = FALSE]

train_nlin_df  <- data.frame(Y = Y_n, T = T_n, C_n)
rhs_nlin       <- paste(colnames(C_n), collapse = " + ")
fmla_nlin      <- if (!nzchar(rhs_nlin)) "Y ~ T" else paste("Y ~ T +", rhs_nlin)
model_ate_nlin <- lm(as.formula(fmla_nlin), data = train_nlin_df)
ate_nlin       <- coef(model_ate_nlin)["T"]

# Unadjusted (naive) regression for comparison
ate_unadj_nlin <- coef(lm(Y_n ~ T_n))["T_n"]

# Bootstrap 95% CI
set.seed(42); n_boot <- 500
ate_boot_nlin <- replicate(n_boot, {
  idx <- sample(nrow(X_nlin_train), replace = TRUE)
  Tb  <- X_nlin_train[idx, T_idx_nlin]
  Yb  <- X_nlin_train[idx, Y_idx_nlin]
  Cb  <- X_nlin_train[idx, conf_nlin, drop = FALSE]
  bdf <- data.frame(Y = Yb, T = Tb, Cb)
  rhs_boot  <- paste(colnames(Cb), collapse = " + ")
  fmla_boot <- if (!nzchar(rhs_boot)) "Y ~ T" else paste("Y ~ T +", rhs_boot)
  coef(lm(as.formula(fmla_boot), data = bdf))["T"]
})
ci_lo_nlin <- quantile(ate_boot_nlin, 0.025)
ci_hi_nlin <- quantile(ate_boot_nlin, 0.975)

# Linear approximation to structural ATE via (I - W)^{-1}
I_nlin <- diag(d_nlin)
struct_ate_nlin <- tryCatch(
  solve(I_nlin - W_best_nlin)[Y_idx_nlin, T_idx_nlin],
  error = function(e) NA_real_
)

cat(sprintf("\nATE  (T=%d → Y=%d)  on Nonlinear SEM\n", T_idx_nlin, Y_idx_nlin))
cat(sprintf("  Adjusted estimate   : %+.4f\n",          ate_nlin))
cat(sprintf("  Unadjusted estimate : %+.4f\n",          ate_unadj_nlin))
cat(sprintf("  Confounding gap     : %+.4f\n",          ate_nlin - ate_unadj_nlin))
cat(sprintf("  Bootstrap 95%% CI   : [%.4f, %.4f]\n",  ci_lo_nlin, ci_hi_nlin))
cat(sprintf("  Structural ATE (W)  : %+.4f  (linear approx.)\n", struct_ate_nlin))

cat("\n--- Validation ---\n")
if (ci_lo_nlin > 0 || ci_hi_nlin < 0) {
  cat("CI excludes 0  --> significant effect at 95%.\n")
} else {
  cat("CI includes 0  --> no strong evidence of effect.\n")
}
if (abs(ate_nlin - ate_unadj_nlin) > 0.05 * abs(ci_hi_nlin - ci_lo_nlin)) {
  cat("Confounders matter: adjusted and unadjusted estimates differ substantially.\n")
} else {
  cat("Limited confounding: adjusted ≈ unadjusted.\n")
}


### Visualization — Adjacency Matrix Heatmaps

Side-by-side heatmaps for all three models on the nonlinear SEM. Denser, more spread-out weights in the linear model are symptomatic of misspecification: the linear SEM tries to compensate for nonlinear variance by spreading edge mass across many variables.


In [ ]:
%%R
make_heatmap <- function(W, title_str, lim = 1) {
  df <- reshape2::melt(W)
  colnames(df) <- c("Effect", "Cause", "Weight")
  ggplot(df, aes(x = Cause, y = Effect, fill = Weight)) +
    geom_tile() +
    scale_fill_gradient2(low = "blue", mid = "white", high = "red",
                         midpoint = 0, limits = c(-lim, lim)) +
    labs(title = title_str, x = "Cause", y = "Effect") +
    theme_minimal() +
    theme(axis.text.x  = element_text(angle = 90, hjust = 1, size = 7),
          axis.text.y  = element_text(size = 7),
          plot.title   = element_text(size = 9, face = "bold"))
}

p1 <- make_heatmap(W_lin_nlin, "Linear NOTEARS\n(nonlinear SEM baseline)")
p2 <- make_heatmap(W_mlp_nlin, "NotearsMLP\n(nonlinear SEM)")
p3 <- make_heatmap(W_sob_nlin, "NotearsSobolev  k = 4\n(nonlinear SEM)")

grid.arrange(p1, p2, p3, ncol = 3)


### Visualization — Learned DAGs

Four side-by-side graph plots: the ground truth and the three model estimates. Visual comparison reveals which edges are recovered correctly, which are spurious, and which are missed.


In [ ]:
%%R
par(mfrow = c(1, 4), mar = c(1, 1, 3, 1))

common_args <- list(
  vertex.label.color = "black",
  edge.arrow.size    = 0.5,
  vertex.size        = 22,
  vertex.label.cex   = 0.85
)

# Ground truth
G_gt <- graph_from_adjacency_matrix(B_true_nlin, mode = "directed")
V(G_gt)$name <- 1:d_nlin
do.call(plot, c(list(G_gt, main = "Ground Truth\n(Nonlinear SEM)",
                     vertex.color = "lightgreen", edge.color = "forestgreen"),
               common_args))

# Linear NOTEARS (misspecified)
G_ln <- graph_from_adjacency_matrix(B_lin_nlin, mode = "directed")
V(G_ln)$name <- 1:d_nlin
do.call(plot, c(list(G_ln, main = "Linear NOTEARS\n(misspecified baseline)",
                     vertex.color = "lightblue", edge.color = "gray40"),
               common_args))

# NotearsMLP
B_plot_mlp <- if (!is.null(acc_mlp_on_nlin)) B_mlp_nlin else matrix(0L, d_nlin, d_nlin)
G_mlp <- graph_from_adjacency_matrix(B_plot_mlp, mode = "directed")
V(G_mlp)$name <- 1:d_nlin
do.call(plot, c(list(G_mlp, main = "NotearsMLP",
                     vertex.color = "lightsalmon", edge.color = "tomato3"),
               common_args))

# NotearsSobolev
B_plot_sob <- if (!is.null(acc_sob_on_nlin)) B_sob_nlin else matrix(0L, d_nlin, d_nlin)
G_sob <- graph_from_adjacency_matrix(B_plot_sob, mode = "directed")
V(G_sob)$name <- 1:d_nlin
do.call(plot, c(list(G_sob, main = "NotearsSobolev  (k = 4)",
                     vertex.color = "plum", edge.color = "purple4"),
               common_args))

par(mfrow = c(1, 1))


### Interpretation

Linear NOTEARS on nonlinear data is a deliberate stress test, not an expected winner. Its higher SHD and lower TPR relative to the nonlinear variants reveal how the linearity assumption inflates structural error even when the optimization converges cleanly.

NotearsMLP tends to outperform the linear baseline when the sample size is sufficient ($n \geq 500$ per variable), at the cost of longer training time and sensitivity to learning rate. NotearsSobolev is competitive with the MLP on the `tanh` and `sin` SEM types used here, because its sine-basis expansion is a natural approximation to those functional forms — and it trains faster.

DAG validity must be verified with `is_dag()` before interpreting any structural metric. The augmented Lagrangian occasionally fails to enforce acyclicity on small nonlinear problems; if this happens, reduce `lr` or increase `max_iter`.


## Summary and Conclusion

NOTEARS transformed causal structure learning by replacing an NP-hard combinatorial search with a smooth continuous optimization problem. Its three-component architecture — a data-fit loss, the smooth acyclicity constraint $h(W) = \mathrm{tr}(e^{W \odot W}) - d$, and an augmented Lagrangian outer loop — is both elegant and extensible: the same scaffold supports linear, nonlinear-MLP, and nonlinear-Sobolev models with minimal code changes.

**Strengths.** The linear variant is simple (< 60 lines), fast on moderate graphs ($d \leq 200$), and produces interpretable weighted edges whose signs indicate excitatory or inhibitory relationships. The nonlinear extensions handle arbitrary smooth functions without forfeiting the acyclicity guarantee.

**Limitations and best practices.** The linear variant assumes additive Gaussian noise with no hidden confounders — violations degrade both structural and effect estimates. Always center or standardize variables before fitting (NOTEARS is scale-sensitive). For nonlinear data, default to NotearsMLP or DAG-GNN / GraN-DAG. Validate on synthetic data with known ground truth before applying to real-world datasets, and cross-validate $\lambda_1$ and $w_{\mathrm{threshold}}$ on held-out SHD when possible.


## Resources

-   **Original Paper**: [DAGs with NO TEARS (Zheng et al., 2018)](https://arxiv.org/abs/1803.01422)
-   **Official Code**: [github.com/xunzheng/notears](https://github.com/xunzheng/notears) — the 60-line reference implementation
-   **Extensions**: gCastle (NOTEARS + many variants), CausalNex (NOTEARS + Bayesian networks)
-   **Further Reading**:
    -   GraN-DAG and DAG-GNN papers for nonlinear extensions
    -   Benchmark datasets: Sachs protein-signaling network, DREAM challenge regulatory networks
-   **Advanced Libraries**: `gcastle`, `causal-learn` (PC/GES), `DoWhy` + `EconML` (downstream effect estimation on learned graphs)
-   **R Packages**: `expm`, `igraph`, `torch`, `ggplot2`, `causaldata`
